# 📱 Will the Customer Switch? — Model Practice

**TeleOne**, a mobile operator, wants to know which customers will move to another operator in the next 3 months, so it can offer them a deal before they leave.

You have data on **600 customers** in `mobile_customers.csv`.

**Your job**
1. Build three models: **KNN**, **Logistic Regression** and **Decision Tree**.
2. For every model, compare **train accuracy** with **test accuracy**.
3. Find which settings **overfit**, which **underfit**, and which give a **good fit**.
4. Pick the model you would give to TeleOne.

Section 1 is already written for you. Run it, then write your own code in every cell marked `# Your code here`.

### 📋 Columns

| Column | Meaning |
|---|---|
| `customer_id` | Customer ID (not useful for prediction) |
| `age` | Age of the customer |
| `plan_type` | `Prepaid` or `Postpaid` |
| `monthly_bill` | Monthly bill in ₹ |
| `data_gb_per_month` | Mobile data used per month (GB) — has some blanks |
| `call_drops_per_week` | Calls dropped per week |
| `months_with_operator` | How long they have been with TeleOne — has some blanks |
| `complaints_last_3m` | Complaints made in the last 3 months |
| `city_tier` | `Tier-1`, `Tier-2` or `Tier-3` city |
| `has_family_plan` | `Yes` or `No` |
| **`switched`** | **Target:** 1 = moved to another operator, 0 = stayed |

### 🔁 Quick recap

| What you see | Name |
|---|---|
| Train accuracy **high**, test accuracy **much lower** | **Overfitting** (the model memorised the training rows) |
| Train accuracy **low** and test accuracy **low** | **Underfitting** (the model is too simple) |
| Train and test both **high** and **close** to each other | **Good fit** |

## ✅ Section 1 — Setup (already done, just run)

> **Flow:** import → load → drop the ID column → split → build the preprocessing step → helper function

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score

sns.set_theme(style="whitegrid")

In [ ]:
df = pd.read_csv("mobile_customers.csv")
print(df.shape)
df.head()

In [ ]:
# Blanks in each column
df.isna().sum()

In [ ]:
# customer_id is only a label, so we drop it. switched is the target.
X = df.drop(columns=["customer_id", "switched"])
y = df["switched"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
print("Train rows:", X_train.shape[0], "| Test rows:", X_test.shape[0])

In [ ]:
num_cols = ["age", "monthly_bill", "data_gb_per_month", "call_drops_per_week",
            "months_with_operator", "complaints_last_3m"]
cat_cols = ["plan_type", "city_tier", "has_family_plan"]

# Number columns: fill blanks with the median, then scale.
# Text columns: one-hot encode.
preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), num_cols),
    ("cat", OneHotEncoder(), cat_cols),
])

In [ ]:
def train_test_accuracy(model):
    """Fit the model on the training data and return (train accuracy, test accuracy)."""
    model.fit(X_train, y_train)
    train_acc = accuracy_score(y_train, model.predict(X_train))
    test_acc = accuracy_score(y_test, model.predict(X_test))
    return round(train_acc, 3), round(test_acc, 3)

# How to use it (from Task 2.2 onward):
# train_acc, test_acc = train_test_accuracy(my_pipeline)

## 🔵 Section 2 — KNN

> **Flow:** build a Pipeline → fit → train vs test accuracy → change `k` → find overfitting and underfitting

**Task 2.1** — Build a `Pipeline` with two steps: `("prep", preprocess)` and `("model", KNeighborsClassifier(n_neighbors=5))`.
Fit it on the training data and print the **train accuracy** and the **test accuracy**. Write this one yourself, without the helper.

In [ ]:
# Your code here


**Task 2.2** — Try `k = 1, 5, 15, 51, 201`. For each `k`, build the pipeline and use `train_test_accuracy()`.
Store the results in a DataFrame called `knn_results` with columns `k`, `train_acc`, `test_acc`.

*Hint:* make an empty list, append a dictionary for each `k`, then `pd.DataFrame(your_list)`.

In [ ]:
# Your code here


**Task 2.3** — Plot train accuracy and test accuracy against `k` on the same chart with `sns.lineplot` (one call for each line, use `marker="o"` and `label=`).

In [ ]:
# Your code here


**Questions** — Which `k` overfits? Which `k` underfits? Which `k` would you choose?

**Your answer:** 

## 🟢 Section 3 — Logistic Regression

> **Flow:** build a Pipeline → train vs test accuracy → confusion matrix → precision and recall

**Task 3.1** — Build a pipeline with `LogisticRegression()` and find its train and test accuracy with `train_test_accuracy()`.

In [ ]:
# Your code here


**Task 3.2** — On the **test** data, make the confusion matrix and show it with `sns.heatmap(..., annot=True, fmt="d")`.
Then print the **precision** and **recall** for customers who switched (class 1).

In [ ]:
# Your code here


**Question** — Is Logistic Regression overfitting? Look at the gap between train and test accuracy.

**Your answer:** 

## 🌳 Section 4 — Decision Tree

> **Flow:** tree with no limit → tree with `max_depth=1` → try every depth → plot → pick the best depth

**Task 4.1** — Build a pipeline with `DecisionTreeClassifier(random_state=42)` (no `max_depth`). Find train and test accuracy.

In [ ]:
# Your code here


**Task 4.2** — Do the same with `max_depth=1`.

In [ ]:
# Your code here


**Task 4.3** — Try `max_depth` from 1 to 12. Store the results in `tree_results` (columns `max_depth`, `train_acc`, `test_acc`) and plot both lines with `sns.lineplot`.

In [ ]:
# Your code here


**Questions** — From your table and plot:
1. What happens to **train** accuracy as `max_depth` grows? What happens to **test** accuracy?
2. Which `max_depth` gives the best test accuracy?

**Your answer:** 

## 📊 Section 5 — Compare all models

> **Flow:** one table with every model → gap = train − test → label each model

**Task 5.1** — Make a DataFrame called `summary` with one row per model and columns `model`, `train_acc`, `test_acc`:

- KNN (k=1)
- KNN (your best k)
- KNN (k=201)
- Logistic Regression
- Decision Tree (no limit)
- Decision Tree (max_depth=1)
- Decision Tree (your best depth)

Then add a column `gap` = `train_acc` − `test_acc`.

*Hint:* you already have most numbers in `knn_results`, `tree_results` and your variables from Sections 3 and 4.

In [ ]:
# Your code here


**Task 5.2** — Add a column `verdict` using these rules (use `.loc` with a condition):

- `gap` more than **0.10** → `"Overfitting"`
- `gap` 0.10 or less **and** `test_acc` below **0.75** → `"Underfitting"`
- everything else → `"Good fit"`

In [ ]:
# Your code here


**Question** — Which model would you give to TeleOne, and why? Use your `summary` table.

**Your answer:** 

## ⭐ Section 6 — Bonus: Random Forest (optional)

We will learn Random Forest in the next class. For now, build it the same way and compare.

**Task 6.1** — Build a pipeline with `RandomForestClassifier(random_state=42)`, find train and test accuracy, and compare it with **Decision Tree (no limit)**.

In [ ]:
# Your code here


**Question** — Both have very high train accuracy. Which one does better on the test data?

**Your answer:** 

## 📝 Section 7 — Quick check

Answer in one line each.

**Q1.** Your teammate at TeleOne trained a model and reports **train accuracy 99%** and **test accuracy 70%**. What is happening, and what would you change in a decision tree?

**Your answer:** 

**Q2.** Another model gives **train accuracy 62%** and **test accuracy 60%**. What is happening?

**Your answer:** 

**Q3.** In KNN, does `k = 1` behave like a very deep tree or like a `max_depth=1` tree? Why?

**Your answer:** 